# Sentiment Analysis of Customer Reviews using NLP and Machine Learning

**Goal**: Classify customer product reviews into three sentiment classes: **Positive**, **Neutral**, and **Negative** using CPU-only traditional Machine Learning and NLP.

### Pipeline Architecture
1. **Data Ingestion & Cleaning**: 3,000 balanced reviews with ratings mapped: 1-2 = negative, 3 = neutral, 4-5 = positive.
2. **Exploratory Data Analysis**: Sentiment distribution, review length distributions, top keywords, and class-specific WordClouds.
3. **Negation-Preserving NLP Preprocessing**: Cleaning, contraction handling, stopword removal preserving negations (`not`, `no`, `never`), and WordNet lemmatization.
4. **Feature Engineering**: TF-IDF vectorization with unigrams and bigrams (`max_features=5000, ngram_range=(1,2)`).
5. **Stratified Split**: 80% Train, 20% Test (random_state=42).
6. **Model Benchmarking**: Multinomial Naive Bayes, Logistic Regression, and Linear SVM.
7. **Hyperparameter Tuning**: 5-fold Stratified GridSearchCV with `class_weight='balanced'`.
8. **Inference**: Prediction function returning sentiment label and calibrated confidence score.

In [ ]:
# 1. Import necessary libraries
import os
import sys
import re
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from wordcloud import WordCloud

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

sns.set_theme(style='whitegrid', font_scale=1.1)
print('All libraries loaded successfully!')

## 1. Load and Inspect Dataset

In [ ]:
# Load the dataset
data_path = os.path.join('..', 'data', 'customer_reviews.csv')
if not os.path.exists(data_path):
    data_path = os.path.join('data', 'customer_reviews.csv')

df = pd.read_csv(data_path)
print(f'Dataset Shape: {df.shape}')
print('\nMissing values:\n', df.isnull().sum())
print('\nDuplicate rows:', df.duplicated().sum())
print('\nClass counts:\n', df['sentiment'].value_counts())
df.head()

## 2. Data Cleaning

In [ ]:
# Drop null and duplicate rows
initial_count = len(df)
df_clean = df.dropna(subset=['review', 'sentiment']).drop_duplicates().reset_index(drop=True)
print(f'Dropped {initial_count - len(df_clean)} rows. Remaining clean rows: {len(df_clean)}')
df = df_clean

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Sentiment distribution countplot
plt.figure(figsize=(7, 4.5), dpi=120)
palette = {'positive': '#2EC4B6', 'neutral': '#FF9F1C', 'negative': '#E71D36'}
ax = sns.countplot(data=df, x='sentiment', palette=palette, edgecolor='black')
plt.title('Sentiment Class Distribution (Balanced)', weight='bold')
for p in ax.patches:
    ax.annotate(str(int(p.get_height())), (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='bottom', weight='bold')
plt.show()

# Review word-count distribution
df['word_count'] = df['review'].astype(str).apply(lambda s: len(s.split()))
plt.figure(figsize=(9, 4.5), dpi=120)
sns.histplot(data=df, x='word_count', hue='sentiment', palette=palette, kde=True, bins=35)
plt.title('Review Word Count Distribution by Class', weight='bold')
plt.xlim(0, df['word_count'].quantile(0.98))
plt.show()

## 4. Negation-Preserving NLP Preprocessing

In [ ]:
lemmatizer = WordNetLemmatizer()
standard_stops = set(stopwords.words('english'))
negations = {'not', 'no', 'never', 'nor', 'neither'}
custom_stops = standard_stops - negations

def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'n\'t\b', ' not', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    tokens = text.split()
    cleaned = [lemmatizer.lemmatize(w) for w in tokens if w not in custom_stops and len(w) > 1]
    return ' '.join(cleaned)

df['cleaned_review'] = df['review'].apply(clean_text)
df = df[df['cleaned_review'].str.strip() != ''].reset_index(drop=True)
print('Sample original: ', df['review'].iloc[0][:100])
print('Sample cleaned:  ', df['cleaned_review'].iloc[0][:100])

## 5. Feature Extraction (TF-IDF with Unigrams and Bigrams)

In [ ]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

X = df['cleaned_review']
y = df['sentiment']

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

X_train_tfidf = tfidf.fit_transform(X_train_raw)
X_test_tfidf = tfidf.transform(X_test_raw)

print(f'TF-IDF Vocabulary: {len(tfidf.vocabulary_)} terms')
print(f'Train shape: {X_train_tfidf.shape}, Test shape: {X_test_tfidf.shape}')

## 6. Train and Evaluate Baseline Machine Learning Models

In [ ]:
baseline_models = {
    'Multinomial Naive Bayes': MultinomialNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Linear SVM': CalibratedClassifierCV(LinearSVC(random_state=42, max_iter=2000))
}

metrics_records = []
for name, model in baseline_models.items():
    model.fit(X_train_tfidf, y_train)
    preds = model.predict(X_test_tfidf)
    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds, average='weighted', zero_division=0)
    rec = recall_score(y_test, preds, average='weighted', zero_division=0)
    f1 = f1_score(y_test, preds, average='weighted', zero_division=0)
    metrics_records.append({'Model': name, 'Accuracy': round(acc, 4), 'Precision': round(prec, 4), 'Recall': round(rec, 4), 'F1': round(f1, 4)})
    print(f'=== {name} ===')
    print(classification_report(y_test, preds, digits=4))

results_table = pd.DataFrame(metrics_records)
results_table

## 7. Hyperparameter Optimization (GridSearchCV)

In [ ]:
# Optimize Logistic Regression with balanced class weights
param_grid = {'C': [0.1, 0.5, 1.0, 2.0, 5.0, 10.0], 'class_weight': ['balanced'], 'solver': ['lbfgs']}
grid = GridSearchCV(LogisticRegression(max_iter=1000, random_state=42), param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid.fit(X_train_tfidf, y_train)

best_estimator = grid.best_estimator_
tuned_preds = best_estimator.predict(X_test_tfidf)
tuned_acc = accuracy_score(y_test, tuned_preds)
print('Best Hyperparameters:', grid.best_params_)
print(f'Tuned Test Accuracy: {tuned_acc:.4f}')

## 8. Real-Time Sample Inference

In [ ]:
sample_reviews = [
    ('This product is totally fantastic! Exceeded my expectations.', 'positive'),
    ('The item is standard, neither good nor bad. Average performance.', 'neutral'),
    ('I did not like this at all. Defective and never worked!', 'negative')
]

for text, expected in sample_reviews:
    cleaned = clean_text(text)
    vec = tfidf.transform([cleaned])
    pred = best_estimator.predict(vec)[0]
    conf = np.max(best_estimator.predict_proba(vec)[0])
    print(f'Review:   "{text}"')
    print(f'Expected: {expected.upper()} | Predicted: {pred.upper()} ({conf*100:.2f}%)\n')